# 05 · The RAG chain end to end

After this notebook you can write the canonical LCEL RAG chain from memory, return sources with the answer, format and
verify citations, make the chain say "I don't know", handle follow-up questions in a conversation, stream the
answer, evaluate retrieval and answers on a small question set, and explain what the framework saved you compared
with the same pipeline in plain Python.

**Time:** ~25 min · **Runs:** offline on CPU in < 1 min · **Needs:** [04 · Loaders, splitters, vector stores and retrievers](04_loaders_splitters_vectorstores_retrievers.ipynb)

## Why this matters in interviews

- "Build a RAG pipeline" is the most common GenAI coding round. With LangChain, the core is five lines — and the
  interviewer then asks for sources, citations, the "I don't know" behaviour, follow-up questions and an evaluation.
- Knowing *which part failed* (retrieval or generation) is the skill they probe: you measure the two separately.
- "What does the framework actually do for you?" — answer with a side-by-side, not an opinion.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `rg00` | Design a RAG system over a knowledge base folder: what language, what packages, and what does the skeleton code look like? |
| `rg01` | What is RAG and why does it exist? |
| `tl15` | Name the pieces of a RAG pipeline and which one usually breaks. |
| `rg24` | How do you verify citations so the model cannot fabricate them? |
| `rg25` | How do you handle empty or low-confidence retrieval? |
| `pr16` | How do you write a prompt that makes the model say "I do not know"? |
| `rg09` | What is query rewriting and what forms does it take? |
| `rg11` | How do you evaluate a RAG system? What do you measure? |
| `rg12` | How do you build a RAG eval set from nothing? |
| `pd10` | Why does LangChain (or any framework) exist, if the providers already have APIs? |

## Setup

Real code: `ChatOpenAI(model="gpt-4.1-mini", temperature=0)` and `OpenAIEmbeddings(model="text-embedding-3-small")`
from `langchain_openai`; `langchain_chat_model()` / `langchain_embeddings()` return those, wired to the active
provider. Offline, the stand-in answers a question-plus-context prompt **extractively**: it returns the context
sentence that best matches the question, with that chunk's `[source]` tag — so citations behave like a careful
model's. A real model paraphrases and combines sentences.

(As in notebook 04, we tell Python that `transformers` / `sentence_transformers` are unavailable so that importing the
text splitters does not drag in PyTorch.)

In [ ]:
import sys
from pathlib import Path
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "llm_setup.py").exists():
        sys.path.insert(0, str(_p)); break
from llm_setup import client, MODEL, EMBED_MODEL, PROVIDER
from llm_setup import langchain_chat_model, langchain_embeddings, OFFLINE

for _heavy in ("sentence_transformers", "transformers"):
    sys.modules.setdefault(_heavy, None)

import re, time, tempfile, warnings
import numpy as np
import matplotlib.pyplot as plt
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda

llm = langchain_chat_model(temperature=0)
emb = langchain_embeddings()

## 1. Index the knowledge base (the notebook 04 recipe, compact)

Four Markdown files → one chunk per `##` section, with the file and heading path in metadata, plus a short `cite` id
(`runbook.md#deploys`) that the model will quote.

In [ ]:
KB = {
"runbook.md": """# Runbook

## Deploys

All production deploys require two approvals and must land before 16:00 UTC on weekdays. Friday deploys are blocked unless the change is tagged hotfix. Every deploy is announced in the releases channel.

## Rollback

To roll back, run `deployctl rollback <service> --to <sha>`. Rollback does not revert database migrations; those need a forward fix. After a rollback, open an incident if customers were affected.
""",
"handbook.md": """# Handbook

## On-call

On-call rotations are one week, Monday 09:00 to Monday 09:00. Acknowledge a page within 5 minutes and escalate to the secondary after 15 minutes.

## Incidents

Any customer-visible outage over 5 minutes needs an incident channel, an incident lead and a written postmortem within 48 hours. The postmortem is blameless and lists follow-up actions with owners.

## Leave

Employees get 25 days of paid holiday per year. Holiday requests go through the HR portal at least two weeks ahead.
""",
"infra.md": """# Infrastructure

## Databases

The primary Postgres cluster runs version 16.2 with a connection pool ceiling of 80 per service. Raising the ceiling requires a capacity review.

## Kubernetes

Clusters are upgraded one minor version at a time, never skipping. Upgrades happen in staging first, with a 72-hour soak.
""",
"security.md": """# Security

## Access

Production database access is read-only by default. Write access is granted per incident and expires automatically after 4 hours.

## Secrets

Secrets and credentials live in the vault and are injected at runtime. A secret committed to git must be rotated even if the commit was reverted.

## Error codes

Error AUTH-4012 means the service token has expired. Request a new token from the vault with `vault token renew`.
""",
}
kb_dir = Path(tempfile.mkdtemp()) / "kb"
kb_dir.mkdir()
for name, text in KB.items():
    (kb_dir / name).write_text(text, encoding="utf-8")

with warnings.catch_warnings():
    warnings.simplefilter("ignore", DeprecationWarning)             # langchain-community sunset notice
    from langchain_community.document_loaders import DirectoryLoader, TextLoader
    from langchain_community.vectorstores import FAISS
from langchain_text_splitters import MarkdownHeaderTextSplitter

docs = DirectoryLoader(str(kb_dir), glob="*.md", loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"}).load()
splitter = MarkdownHeaderTextSplitter(headers_to_split_on=[("#", "doc"), ("##", "section")])
chunks = []
for d in sorted(docs, key=lambda d: d.metadata["source"]):
    source = Path(d.metadata["source"]).name
    for c in splitter.split_text(d.page_content):
        c.metadata.update(source=source, cite=f"{source}#{c.metadata['section'].lower().replace(' ', '-')}")
        chunks.append(c)

vectorstore = FAISS.from_documents(chunks, emb, relevance_score_fn=lambda d: 1 - d / 2)   # cosine, see notebook 04
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print(len(chunks), "chunks:", [c.metadata["cite"] for c in chunks])

## 2. The canonical LCEL RAG chain

```
question ──┬── retriever ── format_docs ──> {context}  ─┐
           └── RunnablePassthrough ──────> {question} ─┴─> prompt ──> llm ──> StrOutputParser ──> answer
```

The dict at the start is a `RunnableParallel`: both branches receive the question string. The prompt is where the
behaviour lives — use only the context, cite, say "I don't know" — so write it carefully and version it.

In [ ]:
def format_docs(docs: list[Document]) -> str:
    """One block per chunk, each starting with its citation tag."""
    return "\n\n".join(f"[{d.metadata['cite']}] {d.page_content}" for d in docs)

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "You answer questions about the engineering handbook. Use only the context below. "
               "Cite the source of every fact in square brackets, like [file.md#section]. "
               "If the context does not contain the answer, say you don't know."),
    ("human", "Context:\n{context}\n\nQuestion: {question}"),
])

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

answer = rag_chain.invoke("Can I deploy on a Friday?")
print(answer)
assert "hotfix" in answer and "[runbook.md#deploys]" in answer

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

def draw_pipeline(steps, title=None):
    """Boxes joined by arrows, left to right."""
    fig, ax = plt.subplots(figsize=(min(2.3 * len(steps), 11), 1.7))
    for i, s in enumerate(steps):
        x = i * 2.3
        ax.add_patch(FancyBboxPatch((x, 0), 1.9, 1, boxstyle="round,pad=0.06", fc="#e7f0ff", ec="#3b5bdb"))
        ax.text(x + 0.95, 0.5, s, ha="center", va="center", fontsize=9)
        if i < len(steps) - 1:
            ax.annotate("", xy=(x + 2.28, 0.5), xytext=(x + 1.98, 0.5), arrowprops=dict(arrowstyle="->", lw=1.6))
    ax.set_xlim(-0.2, len(steps) * 2.3); ax.set_ylim(-0.25, 1.25); ax.axis("off")
    if title: ax.set_title(title)
    plt.show()

draw_pipeline(["question", "retriever\n(top-k chunks)", "format_docs\n[cite] text", "prompt\n(rules + context)",
               "chat model", "StrOutputParser\n→ answer"], title="The LCEL RAG chain")

**Always be able to see the exact prompt.** Run the first half of the chain on its own — this is the first thing to
print when an answer is wrong: was the right chunk in the context at all?

In [ ]:
prompt_value = ({"context": retriever | format_docs, "question": RunnablePassthrough()} | rag_prompt).invoke(
    "Can I deploy on a Friday?")
for m in prompt_value.to_messages():
    print(f"--- {m.type}\n{m.content[:420]}")

## 3. Return the sources with the answer

Users (and your evaluation) need the retrieved documents, not just the text. Keep them in the output with
`RunnableParallel` + `.assign`: retrieve once, then add the answer next to the `context` and `question` keys.

In [ ]:
answer_from_docs = (
    RunnableLambda(lambda x: {"context": format_docs(x["context"]), "question": x["question"]})
    | rag_prompt | llm | StrOutputParser()
)
rag_with_sources = RunnableParallel(context=retriever, question=RunnablePassthrough()).assign(answer=answer_from_docs)

out = rag_with_sources.invoke("What does AUTH-4012 mean?")
print("keys    :", list(out))
print("answer  :", out["answer"])
print("sources :", [d.metadata["cite"] for d in out["context"]])
assert out["context"][0].metadata["cite"] == "security.md#error-codes"

## 4. Citations you can check

A model can invent a citation as easily as a fact. Cheap, effective defence (`rg24`): every tag in the answer must be
one of the chunks you actually retrieved; anything else is flagged (or the answer is regenerated). Stronger checks
compare each cited sentence with the chunk text (overlap, an NLI model, or an LLM judge — see
[RAG evaluation](../12_evaluation/03_rag_evaluation_retrieval_and_faithfulness.ipynb)).

In [ ]:
def verify_citations(answer: str, docs: list[Document]) -> dict:
    cited = re.findall(r"\[([^\[\]]+?\.md#[^\[\]]+?)\]", answer)
    allowed = {d.metadata["cite"] for d in docs}
    return {"cited": cited, "unknown": [c for c in cited if c not in allowed], "has_citation": bool(cited)}

print("real answer :", verify_citations(out["answer"], out["context"]))
fake = out["answer"] + " Tokens can also be renewed by the CTO [security.md#executive-override]."
print("tampered    :", verify_citations(fake, out["context"]))
assert not verify_citations(out["answer"], out["context"])["unknown"]
assert verify_citations(fake, out["context"])["unknown"] == ["security.md#executive-override"]

## 5. "I don't know" — in the prompt and before the prompt

Two layers (`pr16`, `rg25`):

1. **The prompt rule** ("if the context does not contain the answer, say you don't know"). Needed, not sufficient:
   models still answer from general knowledge sometimes.
2. **A retrieval gate**: if no chunk clears a relevance threshold, do not call the LLM at all. Cheaper, and it cannot
   hallucinate. The threshold is corpus- and model-specific: tune it on your eval set. (The threshold retriever also
   logs "No relevant docs were retrieved..." when it comes back empty.)

In [ ]:
print("prompt rule:", rag_chain.invoke("What is the capital of Peru?"))

gated_retriever = vectorstore.as_retriever(search_type="similarity_score_threshold",
                                           search_kwargs={"score_threshold": 0.2, "k": 3})
IDK = "I don't know - nothing in the handbook covers that."

def answer_or_abstain(question: str) -> dict:
    docs = gated_retriever.invoke(question)
    if not docs:
        return {"question": question, "context": [], "answer": IDK}      # no LLM call
    return {"question": question, "context": docs, "answer": answer_from_docs.invoke({"context": docs, "question": question})}

gated = RunnableLambda(answer_or_abstain)
calls_before = OFFLINE.request_count if OFFLINE else None
for q in ["What is the capital of Peru?", "Who can write to the production database?"]:
    r = gated.invoke(q)
    print(f"gate | {q:42s} -> {r['answer'][:75]}")
if OFFLINE:
    print("chat calls made for the two questions:", OFFLINE.request_count - calls_before - 2, "(the other 2 were query embeddings)")
assert gated.invoke("What is the capital of Peru?")["answer"] == IDK

## 6. Conversational RAG: rewrite the follow-up, then retrieve

**In plain English:** "What if it is urgent?" means nothing to a search engine. Before retrieving, ask the model to
rewrite the follow-up into a standalone question using the chat history ("Can I deploy on a Friday if the change is
urgent?"), then retrieve with that (`rg09`). The legacy helper for this is
`langchain_classic.chains.create_history_aware_retriever`; in LCEL it is two small steps.

We ask for the rewrite as structured output so it cannot come back wrapped in chatter. (Offline, the stand-in just
copies the prompt text — history plus follow-up — into the field; crude, but it carries the topic words, which is the
point of the rewrite. A real model returns a clean one-line question.)

In [ ]:
from pydantic import BaseModel, Field

class Standalone(BaseModel):
    """A search query that can be understood without the chat history."""
    standalone_question: str = Field(description="The follow-up rewritten as a self-contained question")

rewrite_prompt = ChatPromptTemplate.from_messages([
    ("system", "Rewrite the user's follow-up as a standalone question that can be understood without the chat history."),
    ("human", "Chat history:\n{history}\n\nFollow-up: {question}"),
])
rewriter = (rewrite_prompt | llm.with_structured_output(Standalone)
            | RunnableLambda(lambda s: " ".join(s.standalone_question.split())))     # one clean line

def format_history(turns: list[tuple[str, str]]) -> str:
    return "\n".join(f"Human: {q}\nAI: {a}" for q, a in turns)

conversational_rag = (
    RunnablePassthrough.assign(standalone=rewriter)
    | RunnablePassthrough.assign(context=lambda x: retriever.invoke(x["standalone"]))
    | RunnablePassthrough.assign(answer=lambda x: answer_from_docs.invoke({"context": x["context"], "question": x["standalone"]}))
)

history = [("Can I deploy on a Friday?", "Friday deploys are blocked unless the change is tagged hotfix.")]
follow_up = "What if it is urgent?"

naive = [d.metadata["cite"] for d in retriever.invoke(follow_up)]
res = conversational_rag.invoke({"history": format_history(history), "question": follow_up})
print("naive retrieval on the follow-up :", naive)
print("after the rewrite                :", [d.metadata["cite"] for d in res["context"]])
print("standalone query (first 90 chars):", repr(res["standalone"][:90]))
print("answer                           :", res["answer"])
assert naive[0] != "runbook.md#deploys" and res["context"][0].metadata["cite"] == "runbook.md#deploys"
assert "hotfix" in res["answer"]

The answer step also receives the standalone question, so the model knows what "it" refers to. Why not just append
the whole history to the query? It works for one turn and degrades fast: old topics pollute the
search, and long histories exceed what an embedding captures. The rewrite costs one extra small LLM call per turn —
use a cheap model for it.

## 7. Streaming the answer

The chain streams because every step passes chunks through (the retriever runs first, then the model's tokens flow
through the parser). Time to first token = retrieval + prompt processing, which is why fast retrieval matters.

In [ ]:
if OFFLINE:
    OFFLINE.latency, OFFLINE.token_delay = 0.2, 0.02

t0, first, pieces = time.perf_counter(), None, []
for piece in rag_chain.stream("How long is an on-call shift?"):
    if first is None and piece:
        first = time.perf_counter() - t0
    pieces.append(piece)
total = time.perf_counter() - t0
if OFFLINE:
    OFFLINE.latency, OFFLINE.token_delay = 0.0, 0.0

print("".join(pieces))
print(f"{len(pieces)} chunks | first token {first:.2f}s | done {total:.2f}s")
assert len(pieces) > 5 and first < total

## 8. A small evaluation: is retrieval finding the right chunk? Is the answer right?

Measure the two halves separately (`rg11`), or you cannot tell *what* broke:

- **Retrieval — hit rate@k:** is the gold chunk among the top k? (plus MRR: 1 / rank of the gold chunk)
- **Generation — answer check:** here a keyword that a correct answer must contain, plus "is every citation real".
  In production: LLM-as-judge for correctness and faithfulness, on a few hundred questions.

Building the set from nothing (`rg12`): write 20–50 real questions with the chunk that answers them, add
no-answer questions, and grow it from production failures.

In [ ]:
EVAL = [  # question, gold chunk, a fact the answer must contain
    ("Can I deploy on a Friday?", "runbook.md#deploys", "hotfix"),
    ("How do I roll back a bad release?", "runbook.md#rollback", "deployctl"),
    ("How long is an on-call shift?", "handbook.md#on-call", "one week"),
    ("What is the connection pool limit?", "infra.md#databases", "80"),
    ("Someone committed a password to git, what now?", "security.md#secrets", "rotated"),
    ("What does AUTH-4012 mean?", "security.md#error-codes", "expired"),
]

rows = []
for q, gold, fact in EVAL:
    r = rag_with_sources.invoke(q)
    cites = [d.metadata["cite"] for d in r["context"]]
    rank = cites.index(gold) + 1 if gold in cites else None
    ok = fact.lower() in r["answer"].lower()
    real_cites = not verify_citations(r["answer"], r["context"])["unknown"]
    rows.append({"q": q, "rank": rank, "ok": ok, "real_cites": real_cites, "answer": r["answer"]})
    print(f"{q:48s} gold rank={rank} answer_ok={ok} citations_real={real_cites}\n    -> {r['answer'][:95]}")

hit_at = {k: np.mean([r["rank"] is not None and r["rank"] <= k for r in rows]) for k in (1, 2, 3)}
mrr = np.mean([1 / r["rank"] if r["rank"] else 0 for r in rows])
answer_acc = np.mean([r["ok"] for r in rows])
print(f"\nhit@1={hit_at[1]:.2f} hit@3={hit_at[3]:.2f} MRR={mrr:.2f} answer accuracy={answer_acc:.2f}")
assert hit_at[3] == 1.0 and all(r["real_cites"] for r in rows) and answer_acc >= 5 / 6

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].bar([f"hit@{k}" for k in hit_at], list(hit_at.values()), color="tab:blue")
ax[0].bar(["MRR"], [mrr], color="tab:purple"); ax[0].bar(["answer acc."], [answer_acc], color="tab:green")
ax[0].set_ylim(0, 1.05); ax[0].set_title("Retrieval vs generation metrics")

# how confident was retrieval? similarity of the gold chunk vs the best wrong chunk, per question
margins = []
for q, gold, _ in EVAL:
    scored = vectorstore.similarity_search_with_relevance_scores(q, k=len(chunks))
    g = next(s for d, s in scored if d.metadata["cite"] == gold)
    wrong = max(s for d, s in scored if d.metadata["cite"] != gold)
    margins.append((g, wrong))
x = np.arange(len(EVAL))
ax[1].bar(x - 0.2, [m[0] for m in margins], 0.4, label="gold chunk")
ax[1].bar(x + 0.2, [m[1] for m in margins], 0.4, label="best wrong chunk")
ax[1].set_xticks(x, [f"Q{i + 1}" for i in x]); ax[1].set_ylabel("cosine similarity")
ax[1].set_title("Retrieval margin per question"); ax[1].legend()
plt.tight_layout(); plt.show()

The margin chart is the one to watch as the corpus grows: a question whose gold chunk barely beats the best wrong one
is the next retrieval failure (offline, that is Q3, the on-call question). Fixes, in order: better chunking and metadata, hybrid search, a reranker, query
rewriting (notebook 04 and [the RAG section](../09_rag/07_advanced_rag_query_rewriting_hyde_multiquery.ipynb)).

## 9. The same pipeline without a framework

The no-framework version (built step by step in [RAG pipeline from scratch](../09_rag/06_rag_pipeline_from_scratch.ipynb))
is short too: embed with the SDK, cosine with numpy, one chat call. Same chunks, same prompt text, same model.

In [ ]:
chunk_texts = [c.page_content for c in chunks]
chunk_vecs = np.array([d.embedding for d in client.embeddings.create(model=EMBED_MODEL, input=chunk_texts).data])

def plain_rag(question: str, k: int = 3) -> tuple[str, list[str]]:
    q = np.array(client.embeddings.create(model=EMBED_MODEL, input=[question]).data[0].embedding)
    sims = chunk_vecs @ q / (np.linalg.norm(chunk_vecs, axis=1) * np.linalg.norm(q))
    top = np.argsort(-sims)[:k]
    context = "\n\n".join(f"[{chunks[i].metadata['cite']}] {chunk_texts[i]}" for i in top)
    system, human = [m.content for m in rag_prompt.format_messages(context=context, question=question)]
    resp = client.chat.completions.create(model=MODEL, temperature=0, messages=[
        {"role": "system", "content": system}, {"role": "user", "content": human}])
    return resp.choices[0].message.content, [chunks[i].metadata["cite"] for i in top]

q = "Who can write to the production database?"
plain_answer, plain_cites = plain_rag(q)
lc = rag_with_sources.invoke(q)
print("plain python:", plain_cites, "->", plain_answer)
print("LangChain   :", [d.metadata["cite"] for d in lc["context"]], "->", lc["answer"])
assert plain_cites == [d.metadata["cite"] for d in lc["context"]]       # identical retrieval

(The plain version reuses `rag_prompt.format_messages` only to guarantee the identical prompt text; in a real
no-framework build it is an f-string.)

### What the framework saved you — and what it did not

| | Plain Python + SDK | LangChain |
|---|---|---|
| core pipeline | ~15 lines | ~5 lines, plus the prompt |
| loaders / splitters / vector stores | write or pick a library for each | one interface, swap FAISS → pgvector in one line |
| streaming, `batch`, async | write it for each step | `.stream()`, `.batch()`, `.ainvoke()` on the whole chain |
| retries, fallbacks, tracing | write them | `.with_retry()`, `.with_fallbacks()`, callbacks / LangSmith |
| sources + answer together | return a tuple | `RunnableParallel(...).assign(answer=...)` |
| seeing the exact prompt | it is right there | you must print it (section 2) |
| debugging | normal Python stack traces | deeper traces through Runnable internals |
| dependencies | `openai`, `numpy` | several `langchain-*` packages that change between major versions |

Honest conclusion: for one small RAG endpoint either is fine and the plain version is easier to reason about; the
framework pays off when you need many integrations, swappable components, streaming/tracing across steps, or when the
flow becomes an agent (notebook 06).

## Try it yourself

**1. Change k.** Rebuild the chain with `k=1`. Does hit rate@1 on `EVAL` change? What is the risk of `k=1` in
general?

In [ ]:
# Solution — try it yourself first, then run this
top1 = vectorstore.as_retriever(search_kwargs={"k": 1})
hits = [top1.invoke(q)[0].metadata["cite"] == gold for q, gold, _ in EVAL]
print("hit@1 with k=1:", np.mean(hits))
assert np.mean(hits) == hit_at[1]      # same ranking, so hit@1 cannot change
print("Risk: with k=1 one ranking mistake leaves the model no chance; multi-part questions need several chunks.")

**2. Add a no-answer question to the eval.** Extend the loop to check that "Who is the CEO of the company?" gets an
"I don't know" from `rag_chain`, and that the gate from section 5 abstains without calling the model.

In [ ]:
# Solution — try it yourself first, then run this
q = "Who is the CEO of the company?"
chain_answer = rag_chain.invoke(q)
gate_answer = gated.invoke(q)["answer"]
print("chain:", chain_answer)
print("gate :", gate_answer)
assert "don't know" in chain_answer.lower() and gate_answer == IDK

**3. Metadata-scoped RAG.** Build a chain that only answers from `security.md` (a filter in `search_kwargs`) and check
that a deploy question now gets security chunks only — and therefore no deploy answer.

In [ ]:
# Solution — try it yourself first, then run this
security_only = vectorstore.as_retriever(search_kwargs={"k": 3, "filter": {"source": "security.md"}})
scoped = RunnableParallel(context=security_only, question=RunnablePassthrough()).assign(answer=answer_from_docs)
r = scoped.invoke("Can I deploy on a Friday?")
print([d.metadata["cite"] for d in r["context"]], "->", r["answer"][:80])
assert all(d.metadata["source"] == "security.md" for d in r["context"])

## Say it in an interview

- **The chain:** `{"context": retriever | format_docs, "question": RunnablePassthrough()} | prompt | llm |
  StrOutputParser()`. Return sources with `RunnableParallel(context=retriever, question=...).assign(answer=...)`.
- **Prompt:** use only the context, cite every fact with the chunk id, say you don't know otherwise. Put the source
  tag on each chunk so citations are cheap to verify.
- **Citations:** every cited id must be a retrieved chunk (string check); faithfulness needs sentence-level checks.
- **Abstain twice:** a prompt rule, and a retrieval gate (no chunk above a tuned threshold → no LLM call).
- **Conversations:** rewrite the follow-up into a standalone question with the history (cheap model), then retrieve.
- **Evaluate halves separately:** retrieval hit rate@k / MRR on a labelled set; answer correctness and faithfulness
  with checks or an LLM judge; include no-answer questions. Watch the gold-vs-best-wrong margin.
- **Usually breaks first:** retrieval (chunking, metadata, missing hybrid search), not the model — print the prompt.
- **Framework vs not:** the core is ~15 lines either way; LangChain buys interchangeable components, streaming,
  batching, retries and tracing on the whole chain; it costs visibility and dependency churn.

## Next

- [06 · Agents and memory](06_agents_and_memory.ipynb) — retrieval as a tool the agent decides to call (agentic RAG).
- [RAG pipeline from scratch](../09_rag/06_rag_pipeline_from_scratch.ipynb) ·
  [advanced RAG: query rewriting, HyDE, multi-query](../09_rag/07_advanced_rag_query_rewriting_hyde_multiquery.ipynb) ·
  [RAG evaluation](../12_evaluation/03_rag_evaluation_retrieval_and_faithfulness.ipynb) ·
  [interview coding round: RAG skeleton](../17_interview_coding_rounds/01_rag_skeleton.ipynb)
- Theory: [LangChain course](../../langchain/index.html) (chapter 8) · [genai_flow course](../../genai_flow/index.html) ·
  [interview bank](../../ai_interview_prep/index.html)